# Logistic Regression: Breast Cancer Tumor Classification

**CS229 Machine Learning - Lecture 3 Implementation**

This notebook implements logistic regression from scratch, following Andrew Ng's CS229 Lecture 3.

---
## 1. Import Libraries & Load Data <a id='section1'></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import roc_curve, auc, precision_recall_curve
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✓ Libraries imported!")

In [ ]:
data = load_breast_cancer()
X = data.data
y = data.target

df = pd.DataFrame(X, columns=data.feature_names)
df['target'] = y

print(df.shape)
df.head()

---
## 2. Exploratory Data Analysis

In [ ]:
df.info()

In [ ]:
df.describe()

---
## 3. Logistic Regression (From Scratch)

In [ ]:
class LogisticRegressionScratch:
    def __init__(self, learning_rate=0.1, n_iterations=1000, method='gradient_ascent'):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.method = method
        self.theta = None
        self.cost_history = []

    def sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def compute_cost(self, X, y, theta):
        m = len(y)
        h = self.sigmoid(X @ theta)
        eps = 1e-7
        return -1/m * np.sum(y*np.log(h+eps) + (1-y)*np.log(1-h+eps))

    def compute_gradient(self, X, y, theta):
        m = len(y)
        h = self.sigmoid(X @ theta)
        return (1/m) * X.T @ (y - h)

    def compute_hessian(self, X, theta):
        m = X.shape[0]
        h = self.sigmoid(X @ theta)
        D = h * (1 - h)
        return -(1/m) * (X.T * D) @ X

    def fit(self, X, y):
        m, n = X.shape
        Xb = np.c_[np.ones((m, 1)), X]
        self.theta = np.zeros(n+1)

        for i in range(self.n_iterations):
            cost = self.compute_cost(Xb, y, self.theta)
            self.cost_history.append(cost)

            grad = self.compute_gradient(Xb, y, self.theta)

            if self.method == 'gradient_ascent':
                self.theta += self.learning_rate * grad

            elif self.method == 'newton':
                H = self.compute_hessian(Xb, self.theta)
                H_inv = np.linalg.pinv(H + 1e-4 * np.eye(len(self.theta)))
                self.theta += H_inv @ grad   # FIXED SIGN

            if (i+1) % 100 == 0:
                print(f"Iter {i+1}: Cost={cost:.4f}")

        return self

    def predict_proba(self, X):
        Xb = np.c_[np.ones((X.shape[0], 1)), X]
        return self.sigmoid(Xb @ self.theta)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

print("✓ LogisticRegressionScratch defined!")

## Prepare Data (2 Features Only)

In [ ]:
selected_features = ['worst perimeter', 'worst concave points']
X2 = df[selected_features].values
y2 = df['target'].values

X_train, X_test, y_train, y_test = train_test_split(X2, y2, test_size=0.2, random_state=42, stratify=y2)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train.shape

## Train: Gradient Ascent

In [ ]:
model_gd = LogisticRegressionScratch(learning_rate=0.1, n_iterations=800, method='gradient_ascent')
model_gd.fit(X_train, y_train)

pred_gd = model_gd.predict(X_test)
acc_gd = accuracy_score(y_test, pred_gd)
acc_gd

## Train: Newton's Method

In [ ]:
model_newton = LogisticRegressionScratch(n_iterations=50, method='newton')
model_newton.fit(X_train, y_train)

pred_newton = model_newton.predict(X_test)
acc_newton = accuracy_score(y_test, pred_newton)
acc_newton

## Compare Convergence

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(model_gd.cost_history, label='Gradient Ascent')
plt.plot(model_newton.cost_history, label="Newton's Method")
plt.legend()
plt.title('Cost Convergence')
plt.xlabel('Iterations')
plt.ylabel('Cost')
plt.show()